# まず、ドキュメントごとのcsvを次々に読むことが目標

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import math, os
def return_json_data(char_sep_datas_path):
    doc_id_list = os.listdir(char_sep_datas_path)
    entire_json_data = {'files':{}}
    for doc_id in doc_id_list:
        if doc_id != '.DS_Store':
            doc_path = os.path.join(char_sep_datas_path, doc_id)
            csv_path = os.path.join(doc_path, f'{doc_id}_coordinate.csv')
            one_doc_json_data = convert_csv_data_to_json(csv_path=csv_path, doc_id=doc_id)
            entire_json_data['files'].update(one_doc_json_data)
    return entire_json_data
# 指定したcsvファイルのデータを目的のjson形式に変換する関数
def convert_csv_data_to_json(csv_path, doc_id):
    return_json_data = {}
    # 画像一枚ごとにアフィニティ計算をするためのデータ
    every_image_cood_dict_for_affinity = get_every_image_cood(csv_path=csv_path)
    for image_id, cood_list in every_image_cood_dict_for_affinity.items():
        affinity_list = calc_affinity_list(cood_list)
        main_region = calc_main_region_list(cood_list)
        json_id = doc_id+'_sep_'+image_id
        return_json_data[json_id] = {
            "main_region": main_region,
            "main_affinity": affinity_list,
            "furi_region" : [],
            "furi_affinity": []
        }
    return return_json_data

def calc_main_region_list(cood_list):
    main_region = []
    for cood in cood_list:
        main_region.append([
            cood[0], cood[1],
            cood[2], cood[1],
            cood[2], cood[3],
            cood[0], cood[3]
        ])
    return main_region

def get_every_image_cood(csv_path):
    one_image_cood_for_affinity = {}
    with open(csv_path, 'r') as f:
        f.readline()
        for line in f:
            one_image_data = line.split(',')
            unicode  = one_image_data[0]
            image_id = one_image_data[1]
            x        = int(one_image_data[2])
            y        = int(one_image_data[3])
            block_id = one_image_data[4]
            char_id  = one_image_data[5]
            width    = int(one_image_data[6])
            height   = int(one_image_data[7].strip())
            if image_id in one_image_cood_for_affinity:
                one_image_cood_for_affinity[image_id].append([x,y, x+width, y+height])
            else:
                one_image_cood_for_affinity[image_id] = [[x,y, x+width, y+height]]
            # json_id = doc_id+'_sep_'+image_id
    return one_image_cood_for_affinity

def calc_affinity_list(
    main_region_list,
    min_horiz_overlap_ratio=0.25,
    max_vertical_gap_ratio=0.6,
    max_vertical_overlap_ratio=0.5,
    nearest_strategy='strict',   # 'strict' or 'relaxed'
    pre_filter_horiz=True,       # True: 水平オーバーラップ>0の候補だけで最近傍探索
    debug=False
):
    """
    各矩形について:
      nearest_strategy='strict'  : 縦方向 gap が最小の 1 個だけを候補にし判定
      nearest_strategy='relaxed' : 下側候補を gap 昇順に評価し最初に条件適合した 1 個を採用
    pre_filter_horiz=True の場合:
        最近傍選定前に “水平オーバーラップ > 0” の矩形だけ候補集合とする
        (列違いの矩形が最近傍として邪魔するケースを軽減)

    返値: [[x0,y0,x1,y1,x2,y2,x3,y3], ...] (上左,上右,下右,下左) を 全て int に丸めて返す。
    """
    rects = []
    for r in main_region_list:
        if len(r) != 4:
            continue
        x0,y0,x1,y1 = r
        if x1 <= x0 or y1 <= y0:
            continue
        rects.append([float(x0), float(y0), float(x1), float(y1)])
    if not rects:
        return []

    # y0 でソート
    rects.sort(key=lambda r: r[1])

    def centroids_lr(x0,y0,x1,y1):
        xc = (x0 + x1)/2.0
        yc = (y0 + y1)/2.0
        left_cx  = (x0 + x0 + xc)/3.0
        left_cy  = (y1 + y0 + yc)/3.0
        right_cx = (x1 + x1 + xc)/3.0
        right_cy = (y0 + y1 + yc)/3.0
        return (left_cx, left_cy, right_cx, right_cy)

    def passes_condition(a, b):
        (x0a,y0a,x1a,y1a) = a
        (x0b,y0b,x1b,y1b) = b
        wa = x1a - x0a; wb = x1b - x0b
        ha = y1a - y0a; hb = y1b - y0b
        avg_h = (ha + hb)/2.0

        # 水平重なり
        horiz_overlap = min(x1a, x1b) - max(x0a, x0b)
        if horiz_overlap <= 0:
            return False, "no horiz overlap"
        horiz_ratio = horiz_overlap / max(1e-6, min(wa, wb))
        if horiz_ratio < min_horiz_overlap_ratio:
            return False, f"horiz_ratio {horiz_ratio:.2f}"

        # 縦関係
        vertical_gap = y0b - y1a
        if vertical_gap > 0:
            if vertical_gap > avg_h * max_vertical_gap_ratio:
                return False, f"gap {vertical_gap:.1f}"
        else:
            overlap_h = min(y1a, y1b) - max(y0a, y0b)
            # 重なり過多判定は無効化済み
        return True, "OK"

    affinities = []
    n = len(rects)

    for i in range(n):
        a = rects[i]
        x0a,y0a,x1a,y1a = a

        # 下側候補収集
        candidates = []
        for j in range(i+1, n):
            b = rects[j]
            x0b,y0b,x1b,y1b = b
            if y1b <= y0a:
                continue  # (理論上不要) 完全に上
            # gap 計算
            if y0b >= y1a:
                gap = y0b - y1a
            else:
                gap = 0.0  # 縦重なり
            if pre_filter_horiz:
                if min(x1a, x1b) - max(x0a, x0b) <= 0:
                    continue
            candidates.append((gap, j))

        if not candidates:
            continue

        # gap 昇順
        candidates.sort(key=lambda x: x[0])

        chosen_index = None

        if nearest_strategy == 'strict':
            gap, j = candidates[0]
            ok, reason = passes_condition(a, rects[j])
            if ok:
                chosen_index = j
            else:
                if debug:
                    print(f"i={i} strict reject j={j} ({reason})")
        else:  # 'relaxed'
            for (gap, j) in candidates:
                ok, reason = passes_condition(a, rects[j])
                if ok:
                    chosen_index = j
                    if debug:
                        print(f"i={i} relaxed pick j={j} gap={gap:.1f}")
                    break
                else:
                    if debug:
                        print(f"i={i} relaxed skip j={j} ({reason})")

        if chosen_index is None:
            continue

        b = rects[chosen_index]
        la_cx, la_cy, ra_cx, ra_cy = centroids_lr(*a)
        lb_cx, lb_cy, rb_cx, rb_cy = centroids_lr(*b)
        # ここで整数近似
        quad_float = [
            la_cx, la_cy,
            ra_cx, ra_cy,
            rb_cx, rb_cy,
            lb_cx, lb_cy
        ]
        quad = [int(round(v)) for v in quad_float]
        affinities.append(quad)

    return affinities


def show_rectang_image(rect_cood_list, show_outline=True, return_array=False):
    """
    rect_cood_list: [[x0,y0,x1,y1], ...]  (座標は画像左上原点想定)
    各長方形領域を1、それ以外0の2Dマップを生成し表示する。
    show_outline: Trueなら枠線を重ね描画
    return_array: Trueなら生成した2D配列(np.ndarray)を返す
    """
    if not rect_cood_list:
        print("rect_cood_list が空です。")
        return np.zeros((1,1), dtype=np.uint8) if return_array else None

    # 座標正規化 & 最大サイズ算出
    max_x = 0
    max_y = 0
    norm_rects = []
    for r in rect_cood_list:
        if len(r) != 4:
            continue
        x0, y0, x1, y1 = r
        x0 = int(np.floor(float(x0)))
        y0 = int(np.floor(float(y0)))
        x1 = int(np.ceil (float(x1)))
        y1 = int(np.ceil (float(y1)))
        # 負座標は0にクリップ
        x0 = max(0, x0); y0 = max(0, y0)
        x1 = max(0, x1); y1 = max(0, y1)
        if x1 <= x0 or y1 <= y0:
            continue
        norm_rects.append((x0, y0, x1, y1))
        max_x = max(max_x, x1)
        max_y = max(max_y, y1)

    if max_x == 0 or max_y == 0:
        print("有効な長方形がありません。")
        return np.zeros((1,1), dtype=np.uint8) if return_array else None

    canvas = np.zeros((max_y, max_x), dtype=np.uint8)

    for (x0, y0, x1, y1) in norm_rects:
        canvas[y0:y1, x0:x1] = 1

    plt.figure(figsize=(6, 6 * (max_y / max_x if max_x else 1)))
    plt.imshow(canvas, cmap='gray', interpolation='nearest')
    plt.title(f"Rect map (count={len(norm_rects)})")
    plt.axis('off')

    if show_outline:
        ax = plt.gca()
        for (x0, y0, x1, y1) in norm_rects:
            w = x1 - x0
            h = y1 - y0
            rect_patch = patches.Rectangle((x0, y0), w, h,
                                           linewidth=0.6, edgecolor='red',
                                           facecolor='none')
            ax.add_patch(rect_patch)

    plt.show()

    if return_array:
        return canvas
    
# ...existing code...

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches

def show_quadrilateral_image(region_list, show_outline=True, return_array=False, fill_value=1):
    """
    region_list: 
        - 長方形  : [x0,y0,x1,y1]
        - 四角形  : [x0,y0,x1,y1,x2,y2,x3,y3] (順番は厳密でなくても可)
      が混在したリスト [[...], [...], ...] を受け取り 2D マップへ描画。
    show_outline: 図示時に境界線を描く
    return_array: True なら生成した2D配列(np.uint8) を返す
    fill_value  : 塗る値 (デフォルト1)
    """
    if not region_list:
        print("region_list が空です。")
        return np.zeros((1,1), dtype=np.uint8) if return_array else None

    # 1. まず全頂点を走査してキャンバスサイズを決める
    max_x = 0
    max_y = 0
    parsed = []  # 各要素: {'type': 'rect'/'quad', 'pts': [(x,y),...]}
    for r in region_list:
        if not isinstance(r, (list, tuple)):
            continue
        if len(r) == 4:
            x0,y0,x1,y1 = r
            try:
                x0=float(x0); y0=float(y0); x1=float(x1); y1=float(y1)
            except:
                continue
            if x1 <= x0 or y1 <= y0:
                continue
            pts = [(x0,y0),(x1,y0),(x1,y1),(x0,y1)]
        elif len(r) == 8:
            try:
                pts = [(float(r[0]), float(r[1])),
                       (float(r[2]), float(r[3])),
                       (float(r[4]), float(r[5])),
                       (float(r[6]), float(r[7]))]
            except:
                continue
        else:
            continue

        # 正規化 (負を0へ)
        norm_pts = []
        for (px,py) in pts:
            px = max(0, px)
            py = max(0, py)
            norm_pts.append((px,py))
            max_x = max(max_x, px)
            max_y = max(max_y, py)

        parsed.append({'pts': norm_pts})

    if max_x <= 0 or max_y <= 0:
        print("有効な領域がありません。")
        return np.zeros((1,1), dtype=np.uint8) if return_array else None

    # 2. キャンバス作成
    W = int(np.ceil(max_x))
    H = int(np.ceil(max_y))
    canvas = np.zeros((H, W), dtype=np.uint8)

    # 3. 塗り潰し
    # OpenCV 利用可能ならポリゴン毎に fill
    try:
        import cv2
        use_cv = True
    except ImportError:
        use_cv = False

    for item in parsed:
        pts = item['pts']
        if use_cv:
            poly = np.array([[int(round(x)), int(round(y))] for (x,y) in pts], dtype=np.int32)
            # fillConvexPoly で十分（四角形想定）。任意凸で問題あれば fillPoly に変更。
            cv2.fillConvexPoly(canvas, poly, fill_value)
        else:
            # フォールバック: 外接矩形で塗り潰し (厳密四角形形状は保持されない)
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            x0 = int(np.floor(min(xs))); x1 = int(np.ceil(max(xs)))
            y0 = int(np.floor(min(ys))); y1 = int(np.ceil(max(ys)))
            if x1 > x0 and y1 > y0:
                canvas[y0:y1, x0:x1] = fill_value

    # 4. 可視化
    plt.figure(figsize=(6, 6 * (H / W if W else 1)))
    plt.imshow(canvas, cmap='gray', interpolation='nearest')
    plt.title(f"Region map (count={len(parsed)})")
    plt.axis('off')

    if show_outline:
        ax = plt.gca()
        for item in parsed:
            pts = item['pts']
            # パスを閉じる
            xs = [p[0] for p in pts] + [pts[0][0]]
            ys = [p[1] for p in pts] + [pts[0][1]]
            ax.plot(xs, ys, color='red', linewidth=0.6)

    plt.show()

    if return_array:
        return canvas

# ...existing code...
import json
from typing import Any, Dict

def save_dict_to_json(
    data: Dict[str, Any],
    out_path: str,
    ensure_ascii: bool = False,
    indent: int = 2,
    overwrite: bool = True,
    mkdir: bool = True
):
    """
    data を out_path に JSON 保存するユーティリティ。
    ensure_ascii=False で UTF-8 そのまま書き出し。
    overwrite=False で既存ファイルがある場合は例外。
    mkdir=True なら親ディレクトリを自動生成。
    """
    out_path = os.path.abspath(out_path)
    parent = os.path.dirname(out_path)
    if mkdir and parent and not os.path.exists(parent):
        os.makedirs(parent, exist_ok=True)
    if (not overwrite) and os.path.exists(out_path):
        raise FileExistsError(f"既に存在: {out_path}")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=indent)
    return out_path


In [2]:
char_sep_datas_path = '../../kuzushiji-recognition/char_sep_datas'
entire_json_data = return_json_data(char_sep_datas_path=char_sep_datas_path)

In [3]:
# Save the entire_json_data dictionary to a JSON file
output_path = os.path.join(char_sep_datas_path, 'gt_json.json')
save_dict_to_json(entire_json_data, output_path)

'c:\\Users\\kotat\\MyPrograms\\MyKuzushiji\\kuzushiji-recognition\\char_sep_datas\\gt_json.json'